In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import copy

# -------------------------------
# DATASET DEFINITION
# -------------------------------
class UCFCrimeDataset(Dataset):
    def __init__(self, base_dir):
        self.files = []
        self.labels = []
        categories = sorted(os.listdir(base_dir))
        self.label_map = {cat: i for i, cat in enumerate(categories)}
        print("Classes mapping:", self.label_map)
        
        for cat in categories:
            cat_path = os.path.join(base_dir, cat)
            if not os.path.isdir(cat_path):
                continue
            for f in os.listdir(cat_path):
                if f.endswith('.npy'):
                    self.files.append(os.path.join(cat_path, f))
                    self.labels.append(self.label_map[cat])
        print(f"Total samples found: {len(self.files)}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(self.files[idx])  
        y = self.labels[idx]
        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# -------------------------------
# MODEL ARCHITECTURE
# -------------------------------
class Attention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Linear(input_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)  
        weighted = x * weights
        return torch.sum(weighted, dim=1) 

class CrimeBiLSTM_Attention(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, num_classes=14):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            num_layers=2,           
            dropout=0.3             
        )
        self.attention = Attention(hidden_dim * 2)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  
        context = self.attention(lstm_out) 
        return self.classifier(context)

# -------------------------------
# EXECUTION SCRIPT
# -------------------------------
def train_bilstm():
    # 1. Setup Device & Data
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")
    
    # Path configured to your local repository folder structure
    DATA_PATH = "../extracted_features" 
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Directory {DATA_PATH} not found. Please verify your working directory.")

    dataset = UCFCrimeDataset(DATA_PATH)
    
    # Train/Validation Split (80/20)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    # FIX: Set num_workers=0 to prevent Windows process spawning crash
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

    # 2. Initialize Model, Loss, Optimizer
    num_classes = len(dataset.label_map)
    model = CrimeBiLSTM_Attention(num_classes=num_classes).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4) 
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # 3. Training Loop with Early Stopping
    EPOCHS = 150
    best_val_loss = float('inf')
    patience_counter = 0
    EARLY_STOPPING_PATIENCE = 15

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                loss = criterion(outputs, y)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct / total
        
        scheduler.step(avg_val_loss)
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_bilstm_model.pth")
            print("  --> Validation loss improved. Model saved!")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    np.save("bilstm_label_map.npy", dataset.label_map)
    print("Training complete. Best model and label map saved successfully.")

if __name__ == "__main__":
    train_bilstm()

Executing on: cuda
Classes mapping: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}
Total samples found: 1610
Epoch [1/150] | Train Loss: 1.9591 | Val Loss: 1.5281 | Val Acc: 50.62%
  --> Validation loss improved. Model saved!
Epoch [2/150] | Train Loss: 1.6052 | Val Loss: 1.8658 | Val Acc: 40.37%
Epoch [3/150] | Train Loss: 1.5063 | Val Loss: 1.6116 | Val Acc: 48.45%
Epoch [4/150] | Train Loss: 1.4322 | Val Loss: 1.4357 | Val Acc: 55.90%
  --> Validation loss improved. Model saved!
Epoch [5/150] | Train Loss: 1.3641 | Val Loss: 1.3176 | Val Acc: 56.83%
  --> Validation loss improved. Model saved!
Epoch [6/150] | Train Loss: 1.2902 | Val Loss: 1.4940 | Val Acc: 45.34%
Epoch [7/150] | Train Loss: 1.2407 | Val Loss: 1.4593 | Val Acc: 52.48%
Epoch [8/150] | Train Loss: 1.1598 | Val Loss: 1.4917 | Val Acc: 50.62%
Epoch [9